# walter

LLM-assisted dataset construction for LASA drugs. *"Walter, are these two LASA drugs? Make no mistakes."*

All parameters, paths, and column names are in **`config.py`** — that is the single source of truth. Edit there, not here.

---

## Nomenclature and Terminology

**Raw registries** $\mathcal{R}^{\text{PH}}_{\text{raw}}$ and $\mathcal{R}^{\text{US}}_{\text{raw}}$ refer to the one-column drug name CSVs for the Philippine FDA and US FDA sources, stored at `_data/R_ph.csv` and `_data/R_us.csv` respectively. Only one is active per run, selected via `DATA_SOURCE` in `config.py`.

**$\mathcal{R}_{\text{clean}}$** is the result of passing the active raw registry through the preprocessing pipeline: lowercased, symbols stripped, duplicates removed. It is an in-memory intermediate only and is not saved to disk. It serves as the sampling pool from which $U$ is drawn.

The final dataset $D$ has columns `x_1, t_1, x_2, t_2, label`, where `x_1` and `x_2` are drug names, `t_1` and `t_2` are their IPA transcriptions, and `label` is the pair's class. $D$ is the union of:

- **$P$** — confirmed LASA pairs. Registry-independent: sourced either from a pre-existing file (`_data/P.csv`, columns `x_1, x_2`) or proposed by a local LLM using $\mathcal{R}_{\text{clean}}$ as its candidate pool. All $P$ rows carry `label = 1`.
- **$U$** — similarity-filtered unlabeled pairs drawn from $\mathcal{R}_{\text{clean}}$ via two-tier sampling. $U$ may contain undetected true LASA pairs. All $U$ rows carry `label = 0` (unlabeled, **not** confirmed negative).

$|U| \gg |P|$, $\quad P \cap U = \emptyset$.

Both $P$ and $U$ are recoverable from $D$ by filtering on `label`. Only $D$ is saved to disk (`_results/D.csv`).

## 1. Project Setup

In [6]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 2. Configuration

All settings are in `config.py`. The cell below just prints the active config so you can confirm before running.

In [7]:
from config import (
    DATA_SOURCE, FROM_FILE,
    UNLABELED_TO_POSITIVE_RATIO, TIER_2_SAMPLE_SIZE, SEED,
    P_INPUT_CSV, D_OUT_CSV,
)

print(f"Data source    : {DATA_SOURCE.name}")
print(f"P from file    : {FROM_FILE}  {'→ ' + str(P_INPUT_CSV) if FROM_FILE else '(LLM proposer)'}")
print(f"U/P ratio      : 1:{UNLABELED_TO_POSITIVE_RATIO}")
print(f"Tier 2 sample  : {TIER_2_SAMPLE_SIZE:,}")
print(f"Seed           : {SEED}")
print()
print(f"Output: D → {D_OUT_CSV}")

Data source    : PH
P from file    : False  (LLM proposer)
U/P ratio      : 1:30
Tier 2 sample  : 10,000
Seed           : 42

Output: D → _results\D.csv


## 3. Preprocessing

Loads the active raw registry (`_data/R_{ph|us}.csv`), normalizes and cleans all drug names
into $\mathcal{R}_{\text{clean}}$. Not saved — in-memory only.

In [8]:
import src.preprocessing as pre

R_clean = pre.run(source=DATA_SOURCE)
print(f"\nCleaned registry: {len(R_clean):,} drug names")
display(R_clean.head(10))

[preprocessing] Source: PH
[preprocessing] Rows: 22,510 raw → 22,456 clean  (dropped 54)

Cleaned registry: 22,456 drug names


,drug_name
0,09 nacl sapher
1,09 sodchlorsaph
2,1 ceeplus
3,1000vc
4,2 gen
5,2 gen 750
6,2 gen scp
7,2 king
8,2 max
9,24 alkaline c


In [5]:
# ad hoc cell
from rapidfuzz import fuzz, process
from src.proposer.prompt import construct_user_prompt
from src.proposer.api_llm import api_response
from config import REGISTRY_COL, LLM_N_PROPOSALS

all_drugs = R_clean[REGISTRY_COL].tolist()
sample_drug = R_clean.sample(n=1, random_state=1)[REGISTRY_COL].iloc[0]
top_matches = process.extract(sample_drug, all_drugs, scorer=fuzz.WRatio, limit=11)
candidates = [m[0] for m in top_matches if m[0] != sample_drug][:10]

user_prompt = construct_user_prompt(sample_drug, "\n".join(candidates), LLM_N_PROPOSALS)
print(f"Target: {sample_drug}\nCandidates: {candidates}\n")

result = api_response(user_prompt, candidates=candidates, debug=True)
print("Cleaned result:", result)

Target: xinprotec
Candidates: ['protec', 'naprotect', 'neprotect', 'prosec', 'xiprox', 'ipratec', 'protecz', 'clotec', 'esotec', 'myotec']


[api_llm] --- RAW MESSAGE DUMP ---
{
  "content": "protec\nnaprotect\nneprotect\nxiprox\nprotecz",
  "refusal": null,
  "role": "assistant",
  "annotations": null,
  "audio": null,
  "function_call": null,
  "tool_calls": null,
  "reasoning_content": "We need to identify look-alike or sound-alike drugs for \"xinprotec\". The target has \"xin\" prefix and \"protec\" suffix. Candidates: protec (missing \"xin\"), naprotect (similar ending but different prefix), neprotect, prosec (sounds like \"protec\" but misspelled), xiprox (similar starting \"xi\" and ending \"x\"), ipratec, protecz (similar to protec with z), clotec, esotec, myotec.\n\nMost likely confusions: \"protec\" is very similar visually and phonetically, missing only \"xin\". \"naprotect\" has \"protec\" embedded. \"neprotect\" similar. \"xiprox\" shares \"xi\" and \"prox\" similar to \"p

## 4. Confirmed LASA Pairs (P)

- `FROM_FILE = True` → reads `_data/P.csv` (must have columns `x_1`, `x_2`)
- `FROM_FILE = False` → generates pairs via the local LLM proposer

Set `FROM_FILE` in `config.py`.

In [9]:
import pandas as pd
from config import FROM_FILE

if FROM_FILE:
    from config import P_INPUT_CSV

    if not P_INPUT_CSV.exists():
        raise FileNotFoundError(
            f"{P_INPUT_CSV} not found. "
            "Place your confirmed LASA pairs CSV there, "
            "or set FROM_FILE = False in config.py to use the LLM proposer."
        )
    P = pd.read_csv(P_INPUT_CSV)
    print(f"Loaded P from {P_INPUT_CSV}: {len(P):,} pairs")
else:
    from config import LLM_OUTPUT_JSON
    from src.proposer.llm import LocalModel
    from src.proposer.inference import run_inference, load_inference
    if not LLM_OUTPUT_JSON.exists():
        run_inference(
            registry_df=R_clean,
            model_choice=LocalModel.QWEN3_1_7B,
        )
    else:
        print(f"[inference] Found existing results at {LLM_OUTPUT_JSON}, skipping inference.")
    P = load_inference(LLM_OUTPUT_JSON)
    print(f"Generated P via LLM: {len(P):,} pairs")

display(P.head())
print("Columns:", list(P.columns))

[inference] Iteration 1: 'njectzole' → njectxone, metzole, jaczole, cezole, cozole
[inference] Iteration 2: 'tioconazole tinidazole' → secnidazole, gonidazole, nidaz, midazolex, godazole
[inference] Iteration 3: 'tracrium' → tracid, atracor, acurium, atrium, dynacurium
[inference] Iteration 4: 'green viii inj' → greenxime, evergreen, gref, ren v, renuvie
[inference] Iteration 5: 'domperone' → domper, dompedone, komperdone, perdone, domperiqo
[inference] Iteration 6: 'glucophage' → glucophage xr, glucophage forte, glimphage, glucopride, glucovance
[inference] Iteration 7: 'aerinex' → cartinex, burinex, acernix, aeruginox, lewinex
[inference] Iteration 8: 'montior l' → montior, montil, monterio, montair, monti plus
[inference] Iteration 9: 'cilvex' → silvex, escivex 10, escivex 5, bilex, celex
[inference] Iteration 10: 'aspishield' → aspifend, shield, alcoshield, micoshield, pureshield
[inference] Iteration 11: 'omecore' → omecare, omacor, ometor, omcare, momecort
[inference] Iteration 1

,x_1,x_2
0,njectzole,njectxone
1,njectzole,metzole
2,njectzole,jaczole
3,njectzole,cezole
4,njectzole,cozole


Columns: ['x_1', 'x_2']


## 5. Unlabeled Pairs (U)

Constructs U via two-tier similarity-filtered sampling.
Parameters come from `config.py` (`UNLABELED_TO_POSITIVE_RATIO`, `TIER_2_SAMPLE_SIZE`, etc.).

In [10]:
import src.noise as noise
from config import UNLABELED_TO_POSITIVE_RATIO, TIER_2_SAMPLE_SIZE, SEED

U = noise.make_noise(
    pairs_df=P,
    registry_df=R_clean,
    ratio=UNLABELED_TO_POSITIVE_RATIO,
    tier_2_sample_size=TIER_2_SAMPLE_SIZE,
    seed=SEED,
)

display(U.head())


[noise] P-vocabulary size   : 2,244
[noise] Known positive pairs: 1,979
[noise] Registry size       : 22,456
[noise] Outside vocab       : 20,212
[noise] Target |U|          : 59,370  (ratio 1:30)
[noise] Similarity threshold: 20 (ANY measure)
[noise] Tier 2 sample size  : 10,000

[noise] Building Tier 1 (anchor-based hard negatives)...
[noise] Tier 1 candidates: 35,787,230
[noise] Building Tier 2 (broader coverage, sample=10,000)...
[noise] Tier 2 candidates: 39,044,466

[noise] Final Tier 1: 38,590
[noise] Final Tier 2: 20,779
[noise] Total U     : 59,369  (actual ratio 1:30.0)


,x_1,x_2,similarity,tier,label
0,cetrocord,levocemax,33.333333,1,0
1,ambesar prime,advil,30.000000,1,0
2,isdin,dekonrex,36.000000,1,0
3,gozolide,anerobizol,44.444444,1,0
4,frixo,hofovir,33.333333,1,0


## 6. Assemble and Save

Cleans, deduplicates, adds IPA transcriptions, and saves `_results/D.csv`.
P and U are recoverable from D by filtering on `label` (1 and 0 respectively).

In [11]:
from src.dataset import assemble_and_save

D = assemble_and_save(P, U, add_phonemes=True)

display(D.head(10))
print(f"\nD shape: {D.shape}")
print(D["label"].value_counts().to_string())

[dataset] Removed 15 duplicate pairs

[dataset] P (clean): 1,979 pairs
[dataset] U (clean): 59,369 pairs
[dataset] D (union, deduped): 61,348 pairs
[dataset]   label=1 (P): 1,979
[dataset]   label=0 (U): 59,369

[dataset] Adding IPA transcriptions...
[phonemes] Unique drug names: 20,839
[phonemes] Batch size       : 256
     256 / 20,839  (1.2%)  [1.6s]
     512 / 20,839  (2.5%)  [1.7s]
     768 / 20,839  (3.7%)  [1.8s]
   1,024 / 20,839  (4.9%)  [1.9s]
   1,280 / 20,839  (6.1%)  [2.0s]
   1,536 / 20,839  (7.4%)  [2.1s]
   1,792 / 20,839  (8.6%)  [2.2s]
   2,048 / 20,839  (9.8%)  [2.3s]
   2,304 / 20,839  (11.1%)  [2.4s]
   2,560 / 20,839  (12.3%)  [2.5s]
   2,816 / 20,839  (13.5%)  [2.6s]
   3,072 / 20,839  (14.7%)  [2.7s]
   3,328 / 20,839  (16.0%)  [2.8s]
   3,584 / 20,839  (17.2%)  [2.9s]
   3,840 / 20,839  (18.4%)  [3.0s]
   4,096 / 20,839  (19.7%)  [3.1s]
   4,352 / 20,839  (20.9%)  [3.3s]
   4,608 / 20,839  (22.1%)  [3.4s]
   4,864 / 20,839  (23.3%)  [3.4s]
   5,120 / 20,839  (2

,x_1,t_1,x_2,t_2,label
0,hibor,hɪbɚ,pregeb 150,pɹɪdʒɛb wʌnhʌndɹɪd fɪfti,0
1,xtracee plus chewtab,ɛkstɹæsiː plʌs tʃuːɾæb,momate az,mɑːmeɪt æz,0
2,retatel plus 40 10,ɹɪteɪɾəl plʌs fɔːɹɾi tɛn,liverprime hd,lɪvɚpɹaɪm eɪtʃdiː,0
3,amisar plus,æmɪsɑːɹ plʌs,profilox,pɹoʊfɪlɑːks,0
4,robitussin dm,ɹɑːbaɪtʌsɪn diːɛm,solmux advance,sɑːlmʌks ɐdvæns,0
5,ponstan sf,pɑːnstən ɛsɛf,ophthamycin,ɑːfθɐmaɪsɪn,0
6,valzepam,væltsɪpæm,valopram,væləpɹæm,1
7,maclar,məklɑːɹ,paradrin forte,pæɹədɹɪn fɔːɹɾeɪ,0
8,gliflomet 500 5,glɪflɑːmɪt faɪvhʌndɹɪd faɪv,boren,bɔːɹən,0
9,altodiab,æltoʊdɪæb,cispa 50,sɪspə fɪfti,0



D shape: (61348, 5)
label
0    59369
1     1979


## 7. LASA-Run Candidate Pairs

Each entry in `_data/lasa_run.json` has:
- `x_1` — the anchor drug name
- `x_2` — confirmed LASA matches (label = 1, already in P)
- `candidates` — the full candidate pool the LLM was shown (10 names)

The candidates **not** in `x_2` are plausible confusibles that were *not* flagged as confirmed LASA.
We add them to D as **label = 0** (unlabeled) so the model can treat them as hard negatives.

Two helpers are provided in `src/dataset.py`:
| Function | What it does |
|---|---|
| `add_lasa_run_unlabeled(lasa_data, D)` | Returns extended D with new label-0 rows appended |
| `write_lasa_run_unlabeled_csv(lasa_data)` | Writes the pairs in isolation to `_results/lasa_run_U.csv` |

In [16]:
# --- 7a. Write unlabeled candidate pairs as a standalone CSV ---
from src.dataset import write_lasa_run_unlabeled_csv

U_lasa = write_lasa_run_unlabeled_csv(lasa_data)
display(U_lasa.head(10))
print(f"\nTotal rows: {len(U_lasa):,}")

[dataset] Removed 50 duplicate pairs
[dataset] Saved unselected-candidate pairs → _results\lasa_run_U.csv  (5,956 rows)


,x_1,x_2,label
0,njectzole,ctz,0
1,njectzole,ole,0
2,njectzole,zo,0
3,njectzole,kzole,0
4,njectzole,mzole,0
5,njectzole,zoled,0
6,njectzole,zolev,0
7,njectzole,cyzole,0
8,njectzole,apzole,0
9,njectzole,azolex,0



Total rows: 5,956


In [15]:
import pandas as pd, json
from config import D_OUT_CSV, LASA_RUN_JSON

D = pd.read_csv(D_OUT_CSV)
with open(LASA_RUN_JSON) as f:
    lasa_data = json.load(f)

In [17]:
# --- 7b. Add unlabeled candidate pairs to D and save ---
# Requires D to already be defined (run Section 6 first, or load from disk via the cell above).
# Set add_phonemes=True to also transcribe the new pairs to IPA.
from src.dataset import add_lasa_run_unlabeled
from config import D_OUT_CSV

D_extended = add_lasa_run_unlabeled(lasa_data, D, add_phonemes=True)

D_extended.to_csv(D_OUT_CSV, index=False)
print(f"Saved → {D_OUT_CSV}  ({len(D_extended):,} rows)")

display(D_extended.tail(10))
print(f"\nD_extended shape: {D_extended.shape}")
print(D_extended['label'].value_counts().to_string())

[dataset] Removed 50 duplicate pairs

[dataset] Unselected-candidate pairs: 5,956
[dataset]   21 already present in D — dropped
[dataset]   5,935 new label=0 rows to add
[phonemes] Unique drug names: 4,471
[phonemes] Batch size       : 256
     256 / 4,471  (5.7%)  [0.1s]
     512 / 4,471  (11.5%)  [0.2s]
     768 / 4,471  (17.2%)  [0.3s]
   1,024 / 4,471  (22.9%)  [0.4s]
   1,280 / 4,471  (28.6%)  [0.5s]
   1,536 / 4,471  (34.4%)  [0.6s]
   1,792 / 4,471  (40.1%)  [0.8s]
   2,048 / 4,471  (45.8%)  [0.9s]
   2,304 / 4,471  (51.5%)  [1.0s]
   2,560 / 4,471  (57.3%)  [1.1s]
   2,816 / 4,471  (63.0%)  [1.2s]
   3,072 / 4,471  (68.7%)  [1.3s]
   3,328 / 4,471  (74.4%)  [1.4s]
   3,584 / 4,471  (80.2%)  [1.5s]
   3,840 / 4,471  (85.9%)  [1.6s]
   4,096 / 4,471  (91.6%)  [1.7s]
   4,352 / 4,471  (97.3%)  [1.8s]
   4,471 / 4,471  (100.0%)  [1.9s]
[phonemes] Done in 1.9s

[dataset] D extended: 61,348 → 67,283 rows
label
0    65304
1     1979
Saved → _results\D.csv  (67,283 rows)


,x_1,t_1,x_2,t_2,label
67273,nicarfree,nɪkɑːɹfɹiː,icare,aɪkɛɹ,0
67274,nicarfree,nɪkɑːɹfɹiː,nurica,njʊɹɹɪkə,0
67275,nicarfree,nɪkɑːɹfɹiː,nasofree,næsəfɹiː,0
67276,nicarfree,nɪkɑːɹfɹiː,norifree,nɔːɹɪfɹiː,0
67277,nicarfree,nɪkɑːɹfɹiː,feme,fiːm,0
67278,nicarfree,nɪkɑːɹfɹiː,fern c sugar free,fɜːn siː ʃʊgɚ fɹiː,0
67279,nicarfree,nɪkɑːɹfɹiː,icaf,aɪkæf,0
67280,nicarfree,nɪkɑːɹfɹiː,ican,aɪkən,0
67281,nicarfree,nɪkɑːɹfɹiː,phree,fɹiː,0
67282,nicarfree,nɪkɑːɹfɹiː,reone,ɹɪoʊn,0



D_extended shape: (67283, 5)
label
0    65304
1     1979


In [18]:
# --- 7c. Patch missing IPA transcriptions in D ---
# Run this if t_1/t_2 are NaN for the new rows (i.e. 7b was previously run with add_phonemes=False).
from src.phonemes import transcribe_dataframe
from config import COL_T1, COL_T2, COL_X1, COL_X2, D_OUT_CSV
import pandas as pd

D_extended = pd.read_csv(D_OUT_CSV)

mask = D_extended[COL_T1].isna() | (D_extended[COL_T1] == '')
print(f'Rows missing transcriptions: {mask.sum():,}')

if mask.sum() > 0:
    # Collect unique drug names that need transcription
    missing_names = pd.unique(
        D_extended.loc[mask, [COL_X1, COL_X2]].values.ravel()
    )
    print(f'Unique names to transcribe: {len(missing_names):,}')

    # Transcribe via a temporary DataFrame
    tmp = pd.DataFrame({COL_X1: missing_names, COL_X2: missing_names})
    tmp = transcribe_dataframe(tmp, verbose=True)
    trans_map = dict(zip(tmp[COL_X1], tmp[COL_T1]))

    # Fill in the missing values
    D_extended.loc[mask, COL_T1] = D_extended.loc[mask, COL_X1].map(trans_map)
    D_extended.loc[mask, COL_T2] = D_extended.loc[mask, COL_X2].map(trans_map)

    D_extended.to_csv(D_OUT_CSV, index=False)
    print(f'Saved → {D_OUT_CSV}  ({len(D_extended):,} rows)')
    display(D_extended[mask].head(10))
else:
    print('Nothing to patch.')


Rows missing transcriptions: 0
Nothing to patch.
